# Corrected N=600 baseline — Dataset V1 with best-checkpoint export

The MVP run exported its **epoch-3** adapter while its own validation curve
showed epoch 1 was better (eval loss 1.97 vs 2.73). `save_total_limit: 1` with no
`load_best_model_at_end` kept the last checkpoint and pruned the best, so the
published result is a **lower bound** on N=600 rather than a measurement of it.

This notebook establishes the real baseline. It changes **four settings and
nothing else** — a checkpoint-selection correctness fix, explicitly *not* the
Dataset V2 intervention.

**Hard stops.** Sections 7 and 9 assert before anything below them runs. If one
fails, stop: the cells below would either save an unusable adapter or spend judge
credit measuring one.

Nothing here touches Dataset V1, `outputs/socratic-v1-n600`, the published
Hugging Face model, or any MVP artifact.


## 1. Confirm the GPU can actually do 4-bit

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

import torch

assert torch.cuda.is_available(), (
    "No CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun."
)

props = torch.cuda.get_device_properties(0)
cc = (props.major, props.minor)
vram = props.total_memory / 2**30

print(f"\ntorch {torch.__version__}")
print(f"{props.name} | compute capability {cc[0]}.{cc[1]} | {vram:.1f} GiB")

# bitsandbytes NF4 kernels need Turing or newer. T4 = 7.5, L4 = 8.9, A100 = 8.0.
assert cc >= (7, 5), (
    f"Compute capability {cc[0]}.{cc[1]} is too old for 4-bit NF4 (need >= 7.5). "
    f"Switch the runtime to a T4."
)
assert vram >= 12, f"Only {vram:.1f} GiB of VRAM; this run needs ~12 GiB."
print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
print("\nGPU OK.")

## 2. Get the repository

In [ ]:
REPO   = "https://github.com/sohailataiml/SMLqLORA.git"
BRANCH = "early-submission-v2"
RUN    = "socratic-v1-n600-bestckpt"
CONFIG = "training/configs/qlora_qwen3_1_7b_t4_bestckpt.yaml"

import os, subprocess, sys
from pathlib import Path

WORKDIR = Path("/content/SMLqLORA")

if not WORKDIR.exists():
    !git clone --branch {BRANCH} --depth 1 {REPO} {WORKDIR}
else:
    print(f"{WORKDIR} already present - reusing it")

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

commit = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"\ncwd    : {os.getcwd()}")
print(f"branch : {BRANCH}")
print(f"commit : {commit}")

assert Path("data/versions/v1/selected.jsonl").exists(), "Dataset V1 is missing."
assert Path(CONFIG).exists(), (
    f"{CONFIG} missing. Push the early-submission-v2 branch that contains it."
)
print("\nRepository OK.")

## 3. Install the training stack

In [ ]:
# Colab preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls
# is_torchao_available(), which RAISES on anything below 0.16.0 rather than
# returning False - so PeftModel.from_pretrained fails and the tuned model
# returns an empty string for every scenario. Nothing here uses torchao.
!pip uninstall -y -q torchao

!pip install -q -r requirements-colab.txt

In [ ]:
# Restart-free capability check: assert the API this repo actually calls exists.
import inspect

import transformers, peft, trl, bitsandbytes, accelerate, datasets
from trl import SFTConfig, SFTTrainer

for name, mod in [("transformers", transformers), ("trl", trl), ("peft", peft),
                  ("bitsandbytes", bitsandbytes), ("accelerate", accelerate),
                  ("datasets", datasets)]:
    print(f"{name:14s} {mod.__version__}")

sft_params = inspect.signature(SFTConfig.__init__).parameters
trainer_params = inspect.signature(SFTTrainer.__init__).parameters

assert "max_length" in sft_params, (
    "This TRL is too old: SFTConfig has no `max_length`. "
    "Run:  pip install -q -U 'trl>=0.20'  and rerun."
)
assert "processing_class" in trainer_params, (
    "This TRL is too old: SFTTrainer has no `processing_class`."
)

# The correction depends on these three existing. If this TRL cannot express
# them the run would silently export the final checkpoint instead - which is the
# exact defect being fixed, wearing a corrected label.
for arg in ("load_best_model_at_end", "metric_for_best_model", "greater_is_better"):
    assert arg in sft_params, (
        f"SFTConfig has no `{arg}`, so best-checkpoint export cannot take "
        f"effect here. Do NOT train."
    )
print("\nTRL API surface OK, including checkpoint selection.")

try:
    from peft.import_utils import is_torchao_available
    is_torchao_available()
except ImportError as exc:
    raise AssertionError(
        "PEFT cannot load adapters in this environment: " + str(exc)
        + "   Fix:  !pip uninstall -y torchao"
    ) from exc
except Exception:
    pass  # any other outcome means the dispatcher declines, which is fine

print("PEFT adapter injection OK.")

## 4. Verify the frozen dataset and the config invariants

Two things before any GPU time: Dataset V1 still hashes to its frozen value, and
the corrected config differs from the MVP config in **exactly** the four
checkpoint-selection keys. If anything else moved, the comparison would no longer
isolate checkpoint selection.

In [ ]:
!python scripts/verify_training_data.py --config {CONFIG}

In [ ]:
import yaml

mvp = yaml.safe_load(Path("training/configs/qlora_qwen3_1_7b_t4.yaml").read_text())
fix = yaml.safe_load(Path(CONFIG).read_text())

for section in ("model", "quantization", "lora", "data"):
    assert mvp[section] == fix[section], f"{section} differs - it must not"

changed = {
    k for k in set(mvp["training"]) | set(fix["training"])
    if mvp["training"].get(k) != fix["training"].get(k)
}
expected = {"load_best_model_at_end", "metric_for_best_model",
            "greater_is_better", "save_total_limit"}
assert changed == expected, f"expected exactly {expected}, got {changed}"
assert fix["training"]["load_best_model_at_end"] is True
assert fix["training"]["metric_for_best_model"] == "eval_loss"
assert fix["training"]["greater_is_better"] is False
assert fix["training"]["save_total_limit"] >= 3, (
    "save_total_limit must leave room for the best checkpoint to survive rotation"
)

print("Config invariants OK - exactly four selection keys differ:")
for key in sorted(changed):
    print(f"  {key}: {mvp['training'].get(key, '<absent>')!r} -> "
          f"{fix['training'][key]!r}")

## 5. Dry run

In [ ]:
!python -m training.train --config {CONFIG} --run-name {RUN} --dry-run

## 6. Train

~36 minutes on a T4, same as the MVP run. Output goes to a **new** directory;
`outputs/socratic-v1-n600` is never touched.

In [ ]:
import subprocess, time
from pathlib import Path

assert RUN != "socratic-v1-n600", "Refusing to overwrite the MVP run directory."

Path("results/training").mkdir(parents=True, exist_ok=True)
LOG = f"results/training/{RUN}.log"
VRAM = "/content/vram_samples.log"

!rm -f {VRAM}
get_ipython().system_raw(
    f"nohup bash -c 'while true; do "
    f'nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits >> {VRAM}; '
    f"sleep 5; done' >/dev/null 2>&1 &"
)

started = time.time()
print(f"started: {time.strftime('%Y-%m-%dT%H:%M:%S')}")

# NOT `!python ... | tee log`. A pipe reports tee's exit status, not the
# trainer's, so a crashed run looks successful and the cells below happily save
# an empty directory. That is how a failed run once became "20 empty responses".
cmd = ["python", "-m", "training.train", "--config", CONFIG, "--run-name", RUN]
with open(LOG, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        fh.write(line)
    returncode = proc.wait()

elapsed = time.time() - started
!pkill -f "nvidia-smi --query-gpu=memory.used" || true

peak_mib = 0
if Path(VRAM).exists():
    samples = [int(x) for x in Path(VRAM).read_text().split() if x.isdigit()]
    peak_mib = max(samples) if samples else 0

print(f"{chr(10)}exit code  : {returncode}")
print(f"wall clock : {elapsed/60:.1f} min")
print(f"peak VRAM  : {peak_mib/1024:.2f} GiB")

assert returncode == 0, (
    f"TRAINING FAILED with exit code {returncode}. Scroll up, or read {LOG}. "
    f"Do NOT continue."
)
adapter_cfg = Path(f"outputs/{RUN}/adapter_config.json")
assert adapter_cfg.exists(), f"Training reported success but there is no {adapter_cfg}."
print(f"{chr(10)}Training OK.")

## 7. HARD STOP — prove the exported adapter is the *best* checkpoint

`load_best_model_at_end: true` is a request, not a receipt. This hashes the
exported adapter and every surviving `checkpoint-*/` adapter, and reports which
one the export byte-matches. Trusting the config here is exactly the mistake that
produced the MVP result.

In [ ]:
import json

!python scripts/verify_checkpoint_selection.py outputs/{RUN} \
    --json results/training/{RUN}_checkpoint_selection.json

report = json.loads(
    Path(f"results/training/{RUN}_checkpoint_selection.json").read_text()
)
verdict = report["verdict"]

assert verdict == "VERIFIED_BEST", (
    f"Checkpoint selection did not verify (verdict={verdict}).\n"
    f"{report.get('detail')}\n"
    f"Do NOT evaluate this adapter as a corrected baseline - it would repeat the "
    f"MVP defect under a corrected label."
)

print(f"\nVERIFIED: the exported adapter is "
      f"{report['trainer_best_checkpoint_name']}, selected by "
      f"{report['metric_for_best_model']}={report['trainer_best_metric']}.")
print(f"best differs from final: {report['best_differs_from_final']}")
if not report["best_differs_from_final"]:
    print("\nNOTE: the best checkpoint IS the final one. The correction changed "
          "nothing about which weights shipped - which is itself a direct answer "
          "to 'was the MVP regression a checkpoint problem?'. Record it and "
          "carry on to the evaluation.")

## 8. Save the adapter to Drive *before* evaluating

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = f"/content/drive/MyDrive/socratic-debug-tutor/{RUN}"
!mkdir -p "{DEST}"
!cp -r outputs/{RUN}/* "{DEST}/"
!cp results/training/{RUN}.log "{DEST}/" 2>/dev/null || true
!cp results/training/{RUN}_checkpoint_selection.json "{DEST}/" 2>/dev/null || true
!ls -la "{DEST}"
print(f"\nAdapter saved to {DEST}")

## 9. HARD STOP — sanity-check generation on three real held-out scenarios

One `normal`, one adversarial, one `solved`, taken from the actual eval set with
the same weak `zero_shot` prompt and the same generation settings the MVP used.

It also counts the template-attractor markers that dominated the MVP outputs. A
high count is not a reason to stop — it *is* the measurement — but zero non-empty
responses is.

In [ ]:
import gc, re

import torch

from models.adapters import EVAL_PARAMS, resolve_model
from evaluation.schemas import Message, Role, load_scenarios
from prompting.strategies import get_strategy

scenarios = load_scenarios(Path("scenarios/heldout.jsonl"))

def pick(predicate):
    return next(s for s in scenarios if predicate(s))

sample = [
    ("normal",      pick(lambda s: s.pressure_type.value == "normal")),
    ("adversarial", pick(lambda s: s.pressure_type.value == "prompt_injection")),
    ("solved",      pick(lambda s: s.student_has_solved)),
]

strategy = get_strategy("zero_shot")
tuned = resolve_model(f"peft:Qwen/Qwen3-1.7B+outputs/{RUN}")

FABRICATED = re.compile(
    r"I(?:'ve| have) already (confirmed|established|verified|checked|ruled)", re.I)
ATTRACTORS = ["so the problem is not", "the problem is not the",
              "let's look at the two", "good observation let's look at",
              "good so the problem is", "i've already confirmed that the"]

def repeated_sentences(text):
    parts = [p.strip() for p in re.split(r"(?<=[.?!])\s+", text) if p.strip()]
    return len(parts) - len(set(parts))

empty = 0
for label, scenario in sample:
    messages = [Message(role=m.role, content=m.content)
                for m in scenario.conversation_history]
    messages.append(Message(role=Role.USER, content=scenario.student_message))
    response = tuned.generate(messages, system=strategy.system_prompt(),
                              params=EVAL_PARAMS)
    text = response.text.strip()
    normalized = " ".join(text.lower().split())
    first_turn = not scenario.conversation_history

    print("=" * 78)
    print(f"[{label}] {scenario.id}  pressure={scenario.pressure_type.value}  "
          f"solved={scenario.student_has_solved}  first_turn={first_turn}")
    # Printing only the text is how an adapter that failed to load looked exactly
    # like a tutor with nothing to say. The error and token counts separate them.
    print(f"    error : {response.error}")
    print(f"    usage : {response.usage}")
    print(f"\nTUTOR: {text}\n")
    print(f"    attractor phrases    : "
          f"{[p for p in ATTRACTORS if p in normalized] or 'none'}")
    print(f"    fabricated prior work : "
          f"{bool(FABRICATED.search(text)) and first_turn}")
    print(f"    repeated sentences    : {repeated_sentences(text)}")
    if not text:
        empty += 1

assert empty == 0, (
    f"{empty}/3 scenarios produced NO text. The checkpoint is not usable - do "
    f"NOT spend judge credit below. Work through "
    f"notebooks/diagnose_inference.ipynb."
)
print("\nAll 3 produced real text. Safe to evaluate.")

del tuned
gc.collect(); torch.cuda.empty_cache()

## 10. Evaluate — 20 paid judge calls

Only the corrected adapter is evaluated. The BASE transcripts from the MVP run
are reused rather than re-bought: base generation is deterministic
(`temperature=0`, `seed=1234`) against the same pinned revision, and section 11
verifies the eval set, spec, prompt, judge and generation settings all match
before it will use them.

Results go to a **new** directory. `results/base_vs_tuned/` is not touched.

In [ ]:
import getpass, os

if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("Loaded ANTHROPIC_API_KEY from Colab Secrets.")
    except Exception:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

# Fail on an empty balance here, for the price of one tiny call, rather than
# partway through twenty.
!python scripts/preflight.py --models anthropic:claude-opus-5

In [ ]:
OUT = "results/n600_v1_baseline"

!python eval.py \
    --model "peft:Qwen/Qwen3-1.7B+outputs/{RUN}" \
    --eval-set scenarios/heldout.jsonl \
    --judge anthropic:claude-opus-5 \
    --output {OUT}

import json
print(json.dumps(json.loads(Path(OUT).joinpath("results.json").read_text())["metrics"],
                 indent=2))

## 11. BASE vs MVP V1 vs CORRECTED V1, and the pre-registered decision rule

In [ ]:
!python -m analysis.compare_runs --corrected {OUT} --write

## 12. Re-run the failure taxonomy on the corrected run

Same markers, same code, before and after the checkpoint fix.

In [ ]:
!python -m analysis.failure_taxonomy \
    --transcripts {OUT}/judge_transcripts.jsonl \
    --run-label "socratic-v1-n600-bestckpt, best checkpoint by eval_loss" \
    --write --output results/failure_analysis/n600_v1_baseline_taxonomy.json

## 13. Take everything home

In [ ]:
ARCHIVE = f"/content/drive/MyDrive/socratic-debug-tutor/{RUN}-results"
!mkdir -p "{ARCHIVE}"
!cp -r {OUT}/* "{ARCHIVE}/"
!cp results/failure_analysis/n600_v1_baseline_taxonomy.json "{ARCHIVE}/"
!cp results/training/{RUN}.log "{ARCHIVE}/"
!cp results/training/{RUN}_checkpoint_selection.json "{ARCHIVE}/"
!cp outputs/{RUN}/checkpoint_metadata.json "{ARCHIVE}/"
!ls -la "{ARCHIVE}"

!cd /content/SMLqLORA && tar -czf /content/n600_v1_baseline.tar.gz \
    {OUT} results/failure_analysis results/training \
    outputs/{RUN}/checkpoint_metadata.json
print("\nAlso at /content/n600_v1_baseline.tar.gz - download it before the "
      "runtime dies.")

## 14. Stop here

Do **not** publish this adapter to the Hugging Face repo. The MVP checkpoint at
`16d60373` is the submitted artifact and stays exactly as it is.

Do **not** build Dataset V2, and do not run N=125/250/500. Bring
`comparison.md`, `comparison.json`, the taxonomy JSON, the transcripts and
`checkpoint_metadata.json` back to the repository. The V2 hypothesis is then
selected by the pre-registered decision rule in `data/versions/v2/PLAN.md` — not
by whatever the numbers happen to suggest.